Pydantic 스키마 정의 : 구조화된 결과 출력

In [1]:
from dotenv import load_dotenv
import os

load_dotenv()

# 사용할 키 가져오기
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# Open AI API 연결
from openai import OpenAI

# 연결된 open ai 객체 생성
client = OpenAI(api_key=OPENAI_API_KEY)

In [2]:
import json
from typing import List
from pydantic import BaseModel, Field

In [3]:
class DataArgs(BaseModel):
    keyword: List[str] = Field(description="추출된 키워드 목록")
    sentiment: str = Field(description="텍스트의 감정 상태 (긍정, 부정, 중립)")
    urgency: int = Field(description="처리 긴급도 (1~5점)")

In [4]:
tools = [
    {
        "type": "function",
        "name": "analyze_customer_feedback",
        "description": "고객의 피드백 내용을 분석하고 결과를 구조화해서 등록합니다.",
        "parameters": DataArgs.model_json_schema()
    }
]

In [5]:
response = client.responses.create(
    model="gpt-5.5",
    input="앱이 결제 중에 자꾸 튕깁니다. 빨리 환불 처리해주세요. 아주 불편합니다.",
    tools=tools
)

In [6]:
response.output

[ResponseReasoningItem(id='rs_0b5e8e0fd70a3d34006a79387d1a1c819a87f219fd6a753182', summary=[], type='reasoning', content=[], encrypted_content='gAAAAABqeTh-QbwoNCpKOCOtmbgRooyh99hK5-U03BhEOBHxhX8nRkqdLlvCY46z4ly816JxTyoaItkuZ2h4qkWa5CAaZgk9pluc1ql04SlP6CskZjNeMndOZTFsPR2GgP-7mCQSXq-zezntVJzq8vtbHSYfwReelGj6cVodggRSIreN9BMqC_eGCZ4gc8yMbsUY17WRilDVad8VKkp6RaVLwZXvwDDBdN2UCiiSC9CX6N3NAOhXFNNvvSkHu1yUcfgqg-3cQnM0Uh8Lbdc1Dj75D71oDSJL1QmRlzlRUPn6T3IX6_ftIMw6SnMFUFRpAjbP_W8PtF0msnV6CgY5gmlNslthy-BhVQvh8Q0mzJ9OEfvkmCBpn1gLZleGkB8hsTqO3_UnpHKNGUI8wT4-Dh688D6-LEf8lsHNLM4av7BPtJCHHSxbiGgWnMWOBd9eYOJOXve9o2vwa9bX6bFzPcwe_Pz6gybLANdgNfYlmESZFi6SRSQnBoYUGPEA3n1AxpHn74MwJrxt5kIMtsr855Kk-tjAEV95Ha3H6gOqtjw09hsrw40kTmIwsWSuNGO8nR6qfys6efot2VZQ3Eh3RTMzy4OhGSGjzIGOqrLq7dPu4JlqcyM5lNmXDk35SU1JUWEy3Ai5Fw0rBszLSanSzSJEmP_lh99K01WuLLtAPwro2KQJ2E8VRBY9OdQ5WAmBeqI1fbbYGoMOOx5Yos_a91gNFL_2uhOhyi0_pQI-25wo4V34jX6FVKCIguzJZywK-MdZZEtn7MV7t81-LbVu_kwPlHukIZXpkKFmwhrziMiXQSDc9rljAGx8erosBvjOA155LYJFGE7A80Hhhh9rXOZ4

In [7]:
for item in response.output:
    if item.type == "function_call":
        result_args = json.loads(item.arguments)
        print("=== 결과 확인 ===")
        print(result_args)

=== 결과 확인 ===
{'keyword': ['앱 튕김', '결제 오류', '환불 요청', '불편'], 'sentiment': '부정', 'urgency': 5}


In [ ]:
# 웹 서치, 파일 서치, 코드 인터프리터
# function calling : 사용자 정의 함수 등록 후 사용(머신러닝, 딥러닝 학습한 모델을 이용해서 함수 정의)
# 출력 결과의 포맷팅 : 파이썬 내장 타입의 객체로 응